> **Kết quả lượng tử hoá đã chốt trên ThinkPad.** Không thuộc lượt train/benchmark FP32 mới trên PC. Mở notebook 02–03 cho luồng mới; không bật đo lại ở notebook này khi train sáu OCR.

# VietLPR_v1: FP32, INT8 tối ưu và INT4 mixed

DET dòng → crop → REC → chuỗi cuối. Bảng gồm FP32, INT8 và INT4 mixed; không ghép YOLO toàn cảnh.

Kernel `.venv-ocr`. Chi tiết: [tối ưu INT8](../docs/int8_optimization.md).

## 1. Môi trường

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/ocr/benchmark.json").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PYTHON = ROOT / ".venv-ocr/Scripts/python.exe"
from ocr.benchmark_display import display_table
from ocr.optimized_benchmark import POINTER
from IPython.display import Markdown, display
print("Python chạy benchmark:", PYTHON)


## 2. Lượt đo chung

Các bảng đọc cùng lượt đo; bảng lượng tử hoá chỉ giữ FP32, INT8 mới và INT4 mixed. FP32 được dùng chung giữa các notebook tương ứng.

Cùng runtime CPU, 4 luồng trên 4 lõi P, OpenCV 1 luồng. Năm vòng độc lập, mỗi model/vòng là tiến trình mới, warm-up 20. Mỗi vòng đủ 124 ảnh test OCR hoặc 258 ảnh detection.

Hiệu chuẩn chỉ train, chọn model chỉ val. Cả ba notebook gọi cùng một bộ điều phối. Chỉ chạy một benchmark tại một thời điểm. Mặc định chỉ đọc kết quả; chạy lại tiếp tục lượt chưa đủ.

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    raise RuntimeError("Đây là hồ sơ lượng tử cũ. Lượt FP32 mới chạy tại notebook 02–03.")
print("Chỉ đọc kết quả đã chốt trên ThinkPad; không đo/train lại.")

## 3. Bảng cuối

Rank theo **FPS giảm dần**. Bảng chỉ giữ FP32, INT8 mới và INT4 mixed. Speedup lấy model chậm nhất trong bảng làm 1×.

Speed ± SD tính từ các trung bình vòng; đọc ảnh, tải model và ghi file ở ngoài vùng đo. Size là file ONNX. Không so trực tiếp FPS ảnh biển với FPS ảnh toàn cảnh. Bảng chỉ công bố sau khi đủ toàn bộ lượt đo.

In [ ]:
# saved_comparison
display(Markdown(display_table("quantization")))
if POINTER.is_file():
    print(POINTER.read_text(encoding="utf-8"))

| Rank | Model | ACC (%) ↑ | CER (%) ↓ | Speed ± SD (ms/ảnh) ↓ | FPS ↑ | Speedup (×) ↑ | Size (MB) ↓ |
|---:|---|---:|---:|---:|---:|---:|---:|
| 1 | VietLPR_v1_INT8_mixed | 94.35 | 0.68 | 1.64 ± 0.28 | 609.85 | 1.08 | 0.84 |
| 2 | VietLPR_v1_FP32 | 94.35 | 0.68 | 1.70 ± 0.32 | 589.96 | 1.05 | 1.23 |
| 3 | VietLPR_v1_INT4_mixed | 94.35 | 0.68 | 1.78 ± 0.28 | 563.32 | 1.00 | 0.77 |

Rank theo FPS giảm dần; model chậm nhất trong bảng làm Speedup 1×.

INT8 dùng lượng tử hoá hỗn hợp; DET/Conv giữ FP32. INT4 cũng là bản mixed.

Run: D:\DUT\NCKH\dataset\runs\int8_optimization\benchmark\20261113_233439
